# CE541E08 — Unit 4 · Day 30 — loc, iloc and DatetimeIndex

| | |
|---|---|
| **Course** | CE541E08 |
| **Department** | Civil Engineering · Christ University |
| **Instructor** | Dr. Arpan Pradhan |
| **Unit** | Unit 4 — The Pandas Library |
| **Session** | Day 30 of 45 |
| **CO** | CO4 |
| **Topics** | loc · iloc · DatetimeIndex · month/year selection · compound filter |

---
> Read the explanation before each code block. Check the expected output. Run the cell and verify. Then try the small challenge at the end.
---

In [ ]:
student_name = "Your Full Name"
roll_number  = "2024XXXXXX"
session      = "Day 30"
print(f"CE541E08 | {student_name} | {roll_number} | {session}")

---
## Section 1 — Selecting Data by Label and Position

Pandas provides two indexers for selecting data:
- **`.loc[label]`** — select by **row label** (index value)
- **`.iloc[n]`** — select by **integer position** (0-based)

A **DatetimeIndex** is an index made of datetime values. It unlocks powerful time-based selection: `df.loc['2024-07']` selects the entire month of July 2024 in one call.

---
## Code Block 1 — .loc and .iloc

### What this code does

We create a small DataFrame with string labels as the index and practice selecting single rows, ranges, and individual values using `.loc` (label-based) and `.iloc` (position-based).

### Why each step is taken

**`.loc['Jul-03']` — single row by label:**
Returns a Series containing all column values for that row. The label must match the index exactly.

**`.loc['Jul-02':'Jul-04']` — slice by label:**
Selects all rows from 'Jul-02' to 'Jul-04' inclusive. Note: unlike Python slicing, `.loc` label slicing **includes** the stop label.

**`.iloc[0]` — first row by position:**
Always selects the first row regardless of what the index label is. Useful when you want the n-th row without knowing its label.

### Algorithm

```
1. Create DataFrame with string date labels as index
2. .loc['Jul-03']       → row with label 'Jul-03'
3. .loc['Jul-02':'Jul-04'] → rows Jul-02, Jul-03, Jul-04 (inclusive)
4. .iloc[0]             → first row by position
```

### Expected output

```
           Flow_m3s  Rainfall_mm   Flag
Jul-01        234.5         45.6   GOOD
Jul-02        678.9         89.3   GOOD
...

.loc 'Jul-03' :
Flow_m3s       1234.5
Rainfall_mm     134.5
Flag             HIGH

.loc range:
           Flow_m3s  Rainfall_mm  Flag
Jul-02        678.9         89.3  GOOD
Jul-03       1234.5        134.5  HIGH
Jul-04        456.7         67.8  GOOD
```

In [ ]:
import pandas as pd

# DataFrame with string index labels (simulated date strings)
stations = pd.DataFrame({
    'Flow_m3s'   : [234.5, 678.9, 1234.5, 456.7, 890.2],
    'Rainfall_mm': [45.6, 89.3, 134.5, 67.8, 112.3],
    'Flag'       : ['GOOD','GOOD','HIGH','GOOD','GOOD'],
}, index=['Jul-01','Jul-02','Jul-03','Jul-04','Jul-05'])

print(stations)
print()

# .loc — select by label
print(".loc 'Jul-03' :"); print(stations.loc['Jul-03'])
print()

# Label slicing with .loc — includes the stop label
print(".loc range:"); print(stations.loc['Jul-02':'Jul-04'])
print()

# .iloc — select by integer position (0-based)
print(".iloc[0] :"); print(stations.iloc[0])

### 🔁 Try this

Use `.iloc[-1]` to select the last row — does it give the same result as `.loc['Jul-05']`?

Try `.iloc[1:3]` — which rows does this give? Remember: `.iloc` slicing **excludes** the stop index.

---
## Code Block 2 — DatetimeIndex: Month Selection

### What this code does

We create a daily DataFrame with a proper `DatetimeIndex` and demonstrate how `pd.date_range` and `.loc` with date strings enable powerful time-based selection.

### Why each step is taken

**`pd.date_range('2024-06-01','2024-08-31',freq='D')`:**
Generates a sequence of daily dates from 1 June to 31 August 2024 (92 days). `freq='D'` means daily frequency. This becomes the index of the DataFrame.

**`df.loc['2024-07']`:**
With a DatetimeIndex, Pandas understands partial date strings. `'2024-07'` selects all rows in July 2024 — all 31 days. Without DatetimeIndex, this would require a complex boolean filter.

### Algorithm

```
1. pd.date_range(start, end, freq='D')
   → 92 daily dates as DatetimeIndex

2. pd.DataFrame({'Flow_m3s':flow}, index=dates)
   → DatetimeIndex as the row labels

3. df.loc['2024-07']
   → selects entire July 2024 (partial string matching on DatetimeIndex)
   → 31 rows returned
```

### Expected output

```
Shape: (92, 1)  Range: 2024-06-01 to 2024-08-31
            Flow_m3s
Date
2024-06-01     ...
...
July records : 31  mean: ... m3/s
```

In [ ]:
import pandas as pd, numpy as np

np.random.seed(42)

# pd.date_range: generates daily dates from start to end
dates = pd.date_range('2024-06-01', '2024-08-31', freq='D')
flow  = np.round(np.random.exponential(300, len(dates)), 1)

# DatetimeIndex as the row index
df = pd.DataFrame({'Flow_m3s': flow}, index=dates)
df.index.name = 'Date'

print(f"Shape: {df.shape}  Range: {df.index[0].date()} to {df.index[-1].date()}")
print(df.head())

# Partial string selection — selects entire month
# Only works because the index is a DatetimeIndex
july = df.loc['2024-07']
print(f"July records : {len(july)}  mean: {july['Flow_m3s'].mean():.1f} m3/s")

### 🔁 Try this

Select all records from the **first two weeks of August** using `.loc['2024-08-01':'2024-08-14']`.

- How many records does this return?
- What is the mean flow in that period?

---
## Code Block 3 — Monsoon vs Dry Season Filter

### What this code does

We create a 5-year daily DataFrame and use `df.index.month` to filter monsoon vs dry-season records — computing mean flow for each period and their ratio.

### Why each step is taken

**`df.index.month`:**
When the index is a DatetimeIndex, `.month` extracts the month number (1=Jan, ..., 12=Dec) for every row. This returns a NumPy-like array that can be used in boolean conditions.

**`(df.index.month>=6)&(df.index.month<=9)`:**
Selects June through September — the Southwest Monsoon season. The result is a boolean array that filters the DataFrame rows.

### Algorithm

```
1. 5-year daily DataFrame (2020-2024)

2. monsoon = df[(index.month>=6) & (index.month<=9)]
   → all Jun-Sep records across 5 years
   → 4 months × ~30 days × 5 years = ~610 records

3. dry = df[(index.month<6)|(index.month>9)]
   → all other months

4. Compute mean and ratio
```

### Expected output

```
Total: 1827, Monsoon: 610, Dry: 1217
Monsoon mean : 490.4 m3/s
Dry mean     : 76.3 m3/s
Ratio        : 6.4x
```

In [ ]:
import pandas as pd, numpy as np

np.random.seed(0)
dates = pd.date_range('2020-01-01', '2024-12-31', freq='D')
base  = np.where((dates.month>=6)&(dates.month<=9), 500, 80)
flow  = np.round(np.maximum(base + np.random.normal(0, base*0.3, len(dates)), 5), 1)
df    = pd.DataFrame({'Flow_m3s':flow}, index=dates)

# df.index.month extracts month number from the DatetimeIndex
# & is element-wise AND — parentheses required around each condition
monsoon = df[(df.index.month>=6) & (df.index.month<=9)]
dry     = df[(df.index.month<6)  | (df.index.month>9)]

print(f"Total: {len(df)}, Monsoon: {len(monsoon)}, Dry: {len(dry)}")
print(f"Monsoon mean : {monsoon['Flow_m3s'].mean():.1f} m3/s")
print(f"Dry mean     : {dry['Flow_m3s'].mean():.1f} m3/s")
print(f"Ratio        : {monsoon['Flow_m3s'].mean()/dry['Flow_m3s'].mean():.1f}x")

### 🔁 Try this

Filter for **intense storm days**: flow > 500 m³/s AND monsoon season.

Use: `df[(df.index.month>=6)&(df.index.month<=9)&(df['Flow_m3s']>500)]`

- How many such days are there across 5 years?
- What fraction of all monsoon days are intense storm days?

---
## Session Summary — loc, iloc and DatetimeIndex

| Selector | Syntax | What it selects |
|---|---|---|
| Label row | `df.loc['label']` | Row with that index label |
| Label range | `df.loc['a':'b']` | Rows a to b inclusive |
| Position row | `df.iloc[n]` | Row at 0-based position n |
| Position range | `df.iloc[n:m]` | Rows n to m-1 |
| Month | `df.loc['2024-07']` | All July 2024 rows (DatetimeIndex) |
| Month number | `df.index.month` | Extract month as integer array |
| Monsoon filter | `df[(df.index.month>=6)&(df.index.month<=9)]` | Jun-Sep rows |

---
## Day 30 Assignment

5-year daily streamflow (already built in Code Block 3):

1. Extract all records for year 2022 using `.loc['2022']`
2. Extract Jul-Aug 2023 using `.loc['2023-07':'2023-08']`
3. Find flood days across all 5 years: flow > 1000 m³/s

### ▶ Assignment cell

In [ ]:
import pandas as pd, numpy as np

np.random.seed(0)
dates=pd.date_range('2020-01-01','2024-12-31',freq='D')
base=np.where((dates.month>=6)&(dates.month<=9),500,80)
flow=np.round(np.maximum(base+np.random.normal(0,base*0.3,len(dates)),5),1)
df=pd.DataFrame({'Flow_m3s':flow},index=dates)

yr2022     = ???   # all 2022 records
jul_aug23  = ???   # Jul-Aug 2023
flood_days = ???   # flow > 1000 m3/s

print(f"2022: {len(yr2022)} records, mean={yr2022['Flow_m3s'].mean():.1f}")
print(f"Jul-Aug 2023: {len(jul_aug23)}")
print(f"Flood days  : {len(flood_days)}")

---
- [ ] Run all cells — verify outputs match expected outputs above
- [ ] Complete the assignment cell (replace `???` placeholders)
- [ ] Upload to GitHub: `Unit4_Pandas/CE541E08_U4_Day30.ipynb`
- [ ] Commit message: `Day 30 assignment completed`

*CE541E08 · Civil Engineering · Christ University · 2026-27 · Dr. Arpan Pradhan*